# AlphaDog — treino do detector de pose canina

Roda no Google Colab com GPU T4 (grátis). ~2 a 4 horas.

**Antes de começar:** Ambiente de execução → Alterar o tipo de ambiente de execução → **T4 GPU** → Salvar.

O que este notebook faz:
1. Confirma que há GPU (sem ela, o treino levaria dias)
2. Recebe o dataset já convertido
3. Treina o YOLO-pose
4. Exporta para TFLite INT8 e ONNX
5. Roda o gate — **sobre o modelo quantizado**, que é o que vai ao celular

## 1. Confirmar a GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        'SEM GPU. Vá em Ambiente de execução -> Alterar o tipo de ambiente '
        '-> T4 GPU -> Salvar, e rode esta célula de novo.\n'
        'Treinar em CPU levaria dias.'
    )

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'memória: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Enviar o dataset

Na sua máquina, compacte a pasta convertida:

```powershell
cd C:\Users\Ramos\Documents\AlphaDog\services\ai\data
Compress-Archive -Path yolo -DestinationPath yolo.zip
```

Dá cerca de 800 MB. Suba para o Google Drive e monte aqui — upload direto pelo
navegador cai no meio do caminho com arquivo desse tamanho.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Ajuste para onde você colocou o zip no Drive.
ZIP = '/content/drive/MyDrive/yolo.zip'

In [ ]:
import zipfile
from pathlib import Path

# Extrai no disco local do Colab, não no Drive: treinar lendo do Drive é 10x
# mais lento e a GPU fica ociosa esperando I/O.
with zipfile.ZipFile(ZIP) as z:
    z.extractall('/content/data')

root = Path('/content/data/yolo')
train = len(list((root / 'images/train').glob('*.jpg')))
val = len(list((root / 'images/val').glob('*.jpg')))
print(f'{train} treino, {val} validação')
assert train > 0 and val > 0, 'dataset vazio — confira o caminho do zip'

In [ ]:
# O yaml aponta para o caminho da sua máquina. Reescreve para o do Colab.
yaml = (root / 'dogs.yaml').read_text()
lines = [
    f'path: {root}' if l.startswith('path:') else l
    for l in yaml.splitlines()
]
(root / 'dogs.yaml').write_text('\n'.join(lines))
print((root / 'dogs.yaml').read_text())

## 3. Instalar

In [ ]:
%pip install -q ultralytics onnx onnxruntime

## 4. Treinar

**nano**, não small ou medium: o alvo é Android intermediário, não flagship.
Um modelo que só roda no top de linha não serve para a maioria dos tutores.

`flipud=0.0` de propósito: cão de cabeça para baixo não existe no uso real e o
augment só ensinaria ruído. `fliplr` usa o `flip_idx` do yaml — sem ele os
lados trocariam, e isso **não aparece na loss**.

In [ ]:
from ultralytics import YOLO

# yolo11n-pose é o checkpoint estável hoje. Se o yolo26n-pose já estiver
# publicado quando você rodar, troque aqui: ele é ~43% mais rápido em CPU.
model = YOLO('yolo11n-pose.pt')

results = model.train(
    data=str(root / 'dogs.yaml'),
    epochs=100,
    imgsz=640,
    batch=32,
    patience=20,
    seed=1337,
    project='/content/runs',
    name='dogpose',
    fliplr=0.5,
    flipud=0.0,
    # Treino acontece em sala, quintal, luz variada. Augment de cor ajuda;
    # geométrico agressivo atrapalha a leitura de pose.
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    # Mosaic corta membros e gera keypoint sem contexto: bom para detecção,
    # ruim para pose. Desligado nas últimas 10 épocas.
    close_mosaic=10,
)

## 5. Exportar

In [ ]:
best = '/content/runs/dogpose/weights/best.pt'
trained = YOLO(best)

# TFLite INT8: é o que vai para o Android. A quantização traz o FPS ao alvo e é
# também onde a acurácia cai — por isso o gate roda sobre ELE, não sobre o .pt.
tflite = trained.export(
    format='tflite', int8=True, imgsz=640, data=str(root / 'dogs.yaml')
)
print('tflite:', tflite)

# ONNX: iOS, e baseline para conferir que a exportação não mudou a saída.
onnx = trained.export(format='onnx', imgsz=640, simplify=True)
print('onnx:', onnx)

## 6. O gate

Mede o modelo **quantizado** contra os critérios decididos antes de existir
modelo — para que não sejam negociados agora que os números apareceram.

O que mais importa não é a acurácia: é o **falso positivo**. Um modelo com 95%
de acerto que mente 5% das vezes vira um "Excelente!" quando o cão não sentou,
e aí o app ensina o tutor a recompensar o comportamento errado.

In [ ]:
!git clone -q https://github.com/CodeByRamos/AlphaDog /content/alphadog
import sys
sys.path.insert(0, '/content/alphadog/services/ai/src')

In [ ]:
import time

import numpy as np
from alphadog_ai.evaluation import Gate, PerformanceMetrics, Sample, build_report
from alphadog_ai.keypoints import BoundingBox, Detection, Keypoint
from alphadog_ai.posture import Posture, classify_posture


def to_detection(result):
    """Converte a saída do Ultralytics para o nosso tipo."""
    if result.boxes is None or len(result.boxes) == 0:
        return None

    # O cão mais confiante do quadro. Múltiplos cães é caso fora do MVP.
    i = int(result.boxes.conf.argmax())
    x1, y1, x2, y2 = result.boxes.xyxy[i].tolist()
    box = BoundingBox(
        x=x1, y=y1, width=x2 - x1, height=y2 - y1,
        confidence=float(result.boxes.conf[i]),
    )

    kp = result.keypoints.data[i]
    points = tuple(
        Keypoint(x=float(p[0]), y=float(p[1]), confidence=float(p[2])) for p in kp
    )
    return Detection(box=box, keypoints=points)


print('Carregue um conjunto de validação rotulado por postura para rodar o gate.')
print('O StanfordExtra tem pose, mas NÃO tem rótulo de sentado/em pé/deitado.')
print('Ver a célula seguinte.')

### O que ainda falta para o gate rodar de verdade

O StanfordExtra tem **keypoints**, mas não tem rótulo de **postura**. Ele diz
onde está a pata; não diz se o cão está sentado.

Então o gate precisa de um conjunto pequeno rotulado à mão:

- ~100 fotos por postura (sentado / em pé / deitado), **incluindo SRD**
- rotuladas por quem sabe o que está olhando

Cem por classe parece pouco e é suficiente: o gate mede *decisão*, não treina
nada. O modelo já aprendeu pose com as 12 mil.

**Enquanto isso não existe, o gate não tem o que julgar** — e sem veredito o
modelo não vai para o app. É de propósito: `SpikeReport.passes()` devolve
`False` quando faltam dados, e spike incompleto não é spike aprovado.

## 7. Baixar o modelo

Salve no Drive antes de fechar o Colab — a sessão apaga tudo ao desconectar,
e você não quer treinar de novo.

In [ ]:
import shutil

dest = Path('/content/drive/MyDrive/alphadog-models')
dest.mkdir(parents=True, exist_ok=True)

for src in [best, str(tflite), str(onnx)]:
    shutil.copy(src, dest / Path(src).name)
    print('salvo:', dest / Path(src).name)

print('\nPróximo passo: baixe o .tflite e coloque em')
print('apps/mobile/assets/models/dogpose.tflite')